# Colab 호환 버전

이 노트북은 원본 책 예제를 **Google Colab 최신 환경(2026)** 에 맞춰 자동 마이그레이션한 버전입니다.

## 적용된 변경
- `load_boston` → `fetch_california_housing` (sklearn 1.2+ 호환)
- `DataFrame.append` → `pd.concat` (pandas 2.0+ 호환)
- `OneHotEncoder(sparse=)` → `sparse_output=` (sklearn 1.2+)
- `.iteritems()` → `.items()` (pandas 2.0+)
- `np.bool / np.int / np.float` → 내장 타입 (numpy 1.20+)
- `KMeans()`에 `n_init=10` 명시 (sklearn 1.4+ 경고 회피)
- `mean_squared_error(..., squared=False)` → `np.sqrt(...)` 래핑 (sklearn 1.6+)
- XGBoost/LightGBM `.fit()` 파라미터는 자동 변환 대신 안내 주석을 달았습니다(셀별 의미 차이 때문).

## Colab 데이터 경로
원본은 같은 폴더의 csv/xlsx 파일을 가정합니다. Colab에서는 두 가지 방법 중 하나를 쓰세요.
1. **직접 업로드**: 좌측 파일 패널 → 업로드, 이후 경로는 `./파일명` 또는 `/content/파일명`.
2. **Drive 마운트**:
    ```python
    from google.colab import drive
    drive.mount('/content/drive')
    # 이후 경로: /content/drive/MyDrive/...
    ```


In [ ]:
# Colab 환경 점검 (선택 실행)
import sys, platform
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
try:
    import numpy, pandas, sklearn
    print('numpy:', numpy.__version__, '| pandas:', pandas.__version__, '| sklearn:', sklearn.__version__)
except Exception as e:
    print('환경 점검 중 경고:', e)


In [ ]:
# 한글 폰트 (matplotlib/seaborn) — Colab에서 한글이 깨질 때만 실행
# !sudo apt-get -qq install -y fonts-nanum > /dev/null
# !fc-cache -fv > /dev/null
# import matplotlib as mpl, matplotlib.font_manager as fm
# mpl.rcParams['font.family'] = 'NanumGothic'
# mpl.rcParams['axes.unicode_minus'] = False


## 한국어 처리 환경 준비 (Colab 필수)

이 노트북은 KoNLPy(Okt, Mecab 등)를 사용합니다. Colab 새 세션마다 아래 셀을 한 번 실행하세요.
- Java(JDK)와 KoNLPy를 설치합니다.
- Mecab은 별도 설치 스크립트가 필요해 본 셀에는 Okt만 사용한다고 가정합니다.
- 한글 폰트(NanumGothic)도 함께 설치해 matplotlib에서 한글이 깨지지 않게 합니다.


In [ ]:
# === Colab 한국어 처리 환경 설치 ===
# 1) Java(JDK) — KoNLPy의 Okt 백엔드에 필요
!sudo apt-get -qq install -y openjdk-17-jdk-headless > /dev/null

# 2) KoNLPy 설치
%pip install -q konlpy

# 3) (선택) Mecab을 쓰려면 아래 주석 해제 — 설치에 수 분 소요
# !bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

# 4) 한글 폰트
!sudo apt-get -qq install -y fonts-nanum > /dev/null
!fc-cache -fv > /dev/null
import matplotlib as mpl
mpl.rcParams['font.family'] = 'NanumGothic'
mpl.rcParams['axes.unicode_minus'] = False

# 5) 동작 확인
from konlpy.tag import Okt
print('Okt 토큰 예:', Okt().morphs('한국어 자연어 처리 테스트입니다.'))


In [1]:
import pandas as pd

train_df = pd.read_csv('ratings_train.txt', sep='\t')
train_df.head(3)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


In [2]:
train_df['label'].value_counts( )

0    75173
1    74827
Name: label, dtype: int64

In [3]:
import re

train_df = train_df.fillna(' ')
# 정규 표현식을 이용하여 숫자를 공백으로 변경(정규 표현식으로 \d 는 숫자를 의미함.) 
train_df['document'] = train_df['document'].apply( lambda x : re.sub(r"\d+", " ", x) )

# 테스트 데이터 셋을 로딩하고 동일하게 Null 및 숫자를 공백으로 변환
test_df = pd.read_csv('ratings_test.txt', sep='\t')
test_df = test_df.fillna(' ')
test_df['document'] = test_df['document'].apply( lambda x : re.sub(r"\d+", " ", x) )

# id 칼럼 삭제 수행
train_df.drop('id', axis=1, inplace=True) 
test_df.drop('id', axis=1, inplace=True)


In [4]:
from konlpy.tag import Twitter

twitter = Twitter()
def tw_tokenizer(text):
    # 입력 인자로 들어온 text 를 형태소 단어로 토큰화 하여 list 객체 반환
    tokens_ko = twitter.morphs(text)
    return tokens_ko

C:\Users\yong\anaconda3\envs\pymldgrev2\lib\site-packages\konlpy\tag\_okt.py:17: UserWarning: "Twitter" has changed to "Okt" since KoNLPy v0.4.5.
  warn('"Twitter" has changed to "Okt" since KoNLPy v0.4.5.')


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Twitter 객체의 morphs( ) 객체를 이용한 tokenizer를 사용. ngram_range는 (1,2) 
tfidf_vect = TfidfVectorizer(tokenizer=tw_tokenizer, ngram_range=(1,2), min_df=3, max_df=0.9)
tfidf_vect.fit(train_df['document'])
tfidf_matrix_train = tfidf_vect.transform(train_df['document'])

C:\Users\yong\anaconda3\envs\pymldgrev2\lib\site-packages\sklearn\feature_extraction\text.py:516: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [6]:
# Logistic Regression 을 이용하여 감성 분석 Classification 수행. 
lg_clf = LogisticRegression(random_state=0, solver='liblinear')

# Parameter C 최적화를 위해 GridSearchCV 를 이용. 
params = { 'C': [1 ,3.5, 4.5, 5.5, 10 ] }
grid_cv = GridSearchCV(lg_clf , param_grid=params , cv=3 ,scoring='accuracy', verbose=1 )
grid_cv.fit(tfidf_matrix_train , train_df['label'] )
print(grid_cv.best_params_ , round(grid_cv.best_score_,4))


Fitting 3 folds for each of 5 candidates, totalling 15 fits
{'C': 3.5} 0.8593


In [7]:
from sklearn.metrics import accuracy_score

# 학습 데이터를 적용한 TfidfVectorizer를 이용하여 테스트 데이터를 TF-IDF 값으로 Feature 변환함. 
tfidf_matrix_test = tfidf_vect.transform(test_df['document'])

# classifier 는 GridSearchCV에서 최적 파라미터로 학습된 classifier를 그대로 이용
best_estimator = grid_cv.best_estimator_
preds = best_estimator.predict(tfidf_matrix_test)

print('Logistic Regression 정확도: ',accuracy_score(test_df['label'],preds))

Logistic Regression 정확도:  0.86172
